# Bayesian Networks and Autoregressive Language Models

This lab connects Bayesian networks with autoregressive language models. We will build a small word-level model from transition counts, test its probability distributions, generate text, and compare first-order and second-order context.

The central idea is the chain rule:

$$P(x_1,\ldots,x_T)=P(x_1)\prod_{t=2}^{T}P(x_t\mid x_1,\ldots,x_{t-1}).$$

**Question 1: Why is this decomposition useful for generating text?**  
It turns a complete-sequence probability into repeated next-token predictions. Once a conditional distribution is available, a sentence can be generated one token at a time.

A simple model approximates the full history with $P(x_t\mid x_{t-1})$. The LLM may help write implementation code, but the probabilistic specification and validation remain our responsibility.

**Question 2: What independence assumption does the first-order network make?**  
It assumes that earlier tokens do not add information once the immediately preceding token is known:

$$P(X_t\mid X_1,\ldots,X_{t-1})=P(X_t\mid X_{t-1}).$$

In [9]:
from collections import Counter, defaultdict
import random

sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

def tokenise(sentence):
    return ["<START>"] + sentence.lower().split() + ["<END>"]

tokenised = [tokenise(sentence) for sentence in sentences]
for sequence in tokenised:
    print(sequence)

['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']


In [10]:
def normalise_counts(counts):
    probabilities = {}
    for context, next_counts in counts.items():
        total = sum(next_counts.values())
        probabilities[context] = {
            token: count / total for token, count in next_counts.items()
        }
    return probabilities

def first_order_counts(data):
    counts = defaultdict(Counter)
    for sequence in data:
        for current, following in zip(sequence, sequence[1:]):
            counts[current][following] += 1
    return counts

counts_1 = first_order_counts(tokenised)
probabilities_1 = normalise_counts(counts_1)

print("Question 3: selected conditional probability tables")
for word in ["the", "cat", "dog", "sat", "ran"]:
    print(f"P(next | {word}) = {probabilities_1.get(word, {})}")

print()
print("Zero-probability transitions are all tokens absent from a row.")
print("For example, P(dog | cat) = 0 and P(mat | cat) = 0.")

print()
print("Question 8: normalization test")
for word, distribution in probabilities_1.items():
    print(f"{word:8} {sum(distribution.values()):.3f}")

Question 3: selected conditional probability tables
P(next | the) = {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
P(next | cat) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
P(next | dog) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
P(next | sat) = {'on': 1.0}
P(next | ran) = {'to': 1.0}

Zero-probability transitions are all tokens absent from a row.
For example, P(dog | cat) = 0 and P(mat | cat) = 0.

Question 8: normalization test
<START>  1.000
the      1.000
cat      1.000
sat      1.000
on       1.000
mat      1.000
rug      1.000
dog      1.000
ran      1.000
to       1.000
park     1.000


## First-order model: Bayesian-network theory

The first-order network is a chain, `X1 -> X2 -> X3 -> ... -> XT`. Transition counts are stored in `counts_1`, and `normalise_counts` converts each row into $P(X_t\mid X_{t-1})$ by dividing by its row total.

**Question 3: What are the conditional probabilities?**  
The code above constructs the required rows. For example, $P(\text{cat}\mid\text{the})=0.25$, $P(\text{dog}\mid\text{the})=0.25$, $P(\text{sat}\mid\text{cat})=2/3$, and $P(\text{ran}\mid\text{cat})=1/3$. Any transition absent from a row has probability zero, such as $P(\text{dog}\mid\text{cat})$.

**Question 4: Where are transition counts stored?**  
They are stored in `counts_1`, a dictionary of `Counter` objects mapping each current token to its observed following tokens and counts.

**Question 5: Where is $P(X_t\mid X_{t-1})$ computed?**  
It is computed by `normalise_counts`; the resulting conditional tables are stored in `probabilities_1`.

**Question 8: What does a normalization total of 0.87 indicate?**  
It indicates an implementation error. Every non-empty conditional row must sum to approximately 1.0, so a total of 0.87 means counts or normalization were handled incorrectly.

The next code cell constructs the tables and checks this invariant.

In [11]:
def choose_next(distribution, mode, rng):
    if mode == "greedy":
        return max(distribution, key=distribution.get)
    return rng.choices(list(distribution), weights=list(distribution.values()), k=1)[0]

def generate_first_order(probabilities, mode="sampling", seed=None, max_length=30):
    rng = random.Random(seed)
    current = "<START>"
    generated = []
    for _ in range(max_length):
        distribution = probabilities.get(current)
        if not distribution:
            break
        current = choose_next(distribution, mode, rng)
        if current == "<END>":
            break
        generated.append(current)
    return " ".join(generated)

print("Question 9: next-word predictions")
for context in ["<START>", "the", "cat", "dog", "sat", "ran"]:
    distribution = probabilities_1.get(context, {})
    if distribution:
        print(context, "->", max(distribution, key=distribution.get), distribution)

generated_20 = [
    generate_first_order(probabilities_1, mode="sampling", seed=100 + i)
    for i in range(20)
]
print()
print("20 saved sampled sentences:")
for i, sentence in enumerate(generated_20, 1):
    print(f"{i:2}. {sentence}")

print()
print("Five greedy sentences:")
for i in range(5):
    print(generate_first_order(probabilities_1, mode="greedy", seed=i))

print()
print("Five sampled sentences:")
for i in range(5):
    print(generate_first_order(probabilities_1, mode="sampling", seed=200 + i))

print()
print("Sampling produces more variation because it can choose any observed continuation; greedy generation always chooses the largest probability.")

Question 9: next-word predictions
<START> -> the {'the': 1.0}
the -> cat {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
cat -> sat {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
dog -> sat {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
sat -> on {'on': 1.0}
ran -> to {'to': 1.0}

20 saved sampled sentences:
 1. the rug
 2. the cat ran to the dog sat on the dog sat on the cat sat on the rug
 3. the dog sat on the mat
 4. the rug
 5. the park
 6. the mat
 7. the park
 8. the rug
 9. the dog ran to the cat sat on the park
10. the rug
11. the mat
12. the cat ran to the dog sat on the cat ran to the mat
13. the dog sat on the dog sat on the mat
14. the park
15. the cat sat on the cat ran to the cat ran to the cat ran to the cat ran to the mat
16. the cat sat on the park
17. the dog ran to the dog sat on the mat
18. the cat sat on the cat sat on the dog sat on the cat sat on the dog sat on the park
19. the cat s

## Generation theory

**Question 6: How does the program choose the next word?**  
Greedy mode selects `argmax`, the token with the largest probability. Sampling mode uses the complete distribution, so lower-probability observed tokens can also be selected.

**Question 7: What happens when no transition has been observed?**  
The generator finds no probability row and stops. This is appropriate for `<END>`; a larger model could instead use smoothing or a shorter fallback context.

**Question 9: Are the most probable predictions always what a person expects?**  
No. They reflect only this small dataset and its limited context, whereas human expectations use broader context, grammar, and world knowledge.

**Question 10: Which generation mode produces more variation?**  
Sampling produces more variation because it can choose any observed continuation. Greedy generation repeatedly chooses the largest probability and may become repetitive or loop, especially in a first-order model.

## Second-order model and comparison

The second-order model changes the conditional distribution to $P(X_t\mid X_{t-2},X_{t-1})$. Its CPT is indexed by pairs of tokens, so it has more context and potentially better predictions, but it needs more data and has more unseen contexts.

**Question 11:** The graph has two parent tokens for each predicted token; the CPT rows are token pairs; the available context increases from one token to two; and the data requirement increases because many more contexts are possible.

**Question 12:** More context can distinguish cases that look identical to a first-order model. However, each additional context creates more possible rows and parameters, so a small dataset leaves many rows unsupported and makes estimates sparse.

In [12]:
def second_order_counts(data):
    counts = defaultdict(Counter)
    for sequence in data:
        for first, second, following in zip(sequence, sequence[1:], sequence[2:]):
            counts[(first, second)][following] += 1
    return counts

def generate_second_order(probabilities, mode="sampling", seed=None, max_length=30):
    rng = random.Random(seed)
    first, second = "<START>", "the"
    generated = [second]
    for _ in range(max_length - 1):
        distribution = probabilities.get((first, second))
        if not distribution:
            break
        following = choose_next(distribution, mode, rng)
        if following == "<END>":
            break
        generated.append(following)
        first, second = second, following
    return " ".join(generated)

counts_2 = second_order_counts(tokenised)
probabilities_2 = normalise_counts(counts_2)

print("Selected second-order distributions:")
for context in [("<START>", "the"), ("the", "cat"), ("cat", "sat"), ("dog", "sat"), ("dog", "ran")]:
    print(context, "->", probabilities_2.get(context, {}))

print("Second-order normalization errors:", [
    context for context, distribution in probabilities_2.items()
    if abs(sum(distribution.values()) - 1.0) > 1e-12
])

first_generated = [generate_first_order(probabilities_1, seed=300 + i) for i in range(30)]
second_generated = [generate_second_order(probabilities_2, seed=300 + i) for i in range(30)]

print()
print("Comparison")
print("first-order rows:", len(probabilities_1), "parameters:", sum(map(len, probabilities_1.values())))
print("second-order rows:", len(probabilities_2), "parameters:", sum(map(len, probabilities_2.values())))
print("first-order distinct outputs:", len(set(first_generated)))
print("second-order distinct outputs:", len(set(second_generated)))
print()
print("First-order examples:")
for sentence in first_generated[:5]:
    print(" ", sentence)
print("Second-order examples:")
for sentence in second_generated[:5]:
    print(" ", sentence)

Selected second-order distributions:
('<START>', 'the') -> {'cat': 0.5, 'dog': 0.5}
('the', 'cat') -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
('cat', 'sat') -> {'on': 1.0}
('dog', 'sat') -> {'on': 1.0}
('dog', 'ran') -> {'to': 1.0}
Second-order normalization errors: []

Comparison
first-order rows: 11 parameters: 17
second-order rows: 14 parameters: 18
first-order distinct outputs: 16
second-order distinct outputs: 6

First-order examples:
  the mat
  the dog sat on the dog sat on the park
  the rug
  the park
  the rug
Second-order examples:
  the dog sat on the rug
  the cat ran to the park
  the dog sat on the mat
  the cat ran to the park
  the cat sat on the mat


## Modern autoregressive language models and reflection

A modern autoregressive language model has the same next-token objective, but estimates the conditional distribution with a neural network rather than a small hand-built CPT. The context is larger, the parameters are learned by gradient-based training, and the model is much more expressive. The shared idea is still:

$$P(\text{next token}\mid\text{previous tokens}).$$

**Question 13: Why is a precise LLM prompt preferable?**  
It specifies the intended variables, representation, probability formula, and generation behaviour. That makes the generated code inspectable and testable. A vague request might produce a neural model, greedy decoding, or a different probability model.

**Question 14: What does the Bayesian-network view add?**  
It makes dependencies explicit, factorises the joint probability, gives meaning to each conditional probability, provides a principled sampling procedure, exposes independence assumptions, and makes the effect of adding context easy to analyse.

The LLM can assist with routine implementation, but it does not replace understanding the intended model. In this lab, normalization tests and inspection of the count structures validate that the implementation matches the specified probability model.

The completed work contains first- and second-order implementations, selected CPTs, normalization tests, generated examples, a model comparison, and this reflection.